# Data Viualisation - Analysis Report

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly as pl
import nbformat
import numpy as np
import functions as fun

In [ ]:
countries_df = pd.read_csv('../../Data/DataFinal/countries_v2.csv')
genres_df = pd.read_csv('../../Data/DataFinal/genres_v2.csv')
languages_df = pd.read_csv('../../Data/DataFinal/languages_v2.csv')
movies_df = pd.read_csv('../../Data/DataFinal/movies_v2.csv', parse_dates=['date'], low_memory=False)
oscar_df =  pd.read_csv('../../Data/DataFinal/the_oscar_awards_v2.csv')
reviews_df = pd.read_csv('../../Data/DataFinal/rotten_tomatoes_reviews_v2.csv', parse_dates=['review_date'])

In [ ]:
plt.style.use('ggplot')

In this phase we will analyse in details the singular features of the datasets and their correlations, and we will try to answer the questions we formulated through the use of graphics representations. Looking at graphics provides prospective on our hypothesis, allowing us to confirm or discard them.

## Which Countries made the most acclamated films?

### Which Countries have the most Oscar nominations?

COMMENTI

In [ ]:
countries_movies_df = fun.countries_movies_merge(countries_df, movies_df)
countries_oscar_df = fun.countries_oscar_merge(countries_movies_df, oscar_df)

In [ ]:
# Count how many times each country appears in the merged DataFrame
country_counts = countries_oscar_df['country'].value_counts()

temp_df = country_counts.reset_index()
temp_df.columns = ['country', 'count']

fig = px.choropleth(
    temp_df,
    locations="country",
    locationmode="country names",
    color="count",
    color_continuous_scale="Viridis",
    range_color=[0, 5000],
    labels={'count': 'Oscar Nominations'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

### Which Countries have the most Oscar wins?

In [ ]:
# Filter only rows where 'winner' is True
winners_only_df = countries_oscar_df[countries_oscar_df['winner'] == True]
country_counts = winners_only_df['country'].value_counts()

temp_df = country_counts.reset_index()
temp_df.columns = ['country', 'count']

fig = px.choropleth(
    temp_df,
    locations="country",
    locationmode="country names",
    color="count",
    color_continuous_scale="Viridis",
    labels={'count': 'Oscar Wins'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

### Correlation between Oscar nominations and wins by Country:

In [ ]:
#nominees_wins_df = fun.countries_oscar_merged(movies_df, countries_df, oscar_df)

nominations_per_country = countries_oscar_df.groupby('country').size().reset_index(name='nominations')
wins_per_country = countries_oscar_df.groupby('country')['winner'].sum().reset_index(name='wins')
country_summary = pd.merge(nominations_per_country, wins_per_country, on='country')

# Scatter plot
fig = px.scatter(
    country_summary,
    x='nominations',
    y='wins',
    size='nominations',
    color='wins',
    hover_name='country',
    labels={'nominations': 'Number of Nominations', 'wins': 'Number of Wins'},
)
fig.update_layout(
    margin={"r":10,"t":40,"l":10,"b":40},
    xaxis_title="Number of Nominations",
    yaxis_title="Number of Wins",
)
fig.show()


As we could immagine, there is a direct correlation between the number of Oscar nominations and wins, with the USA easily being first in both categories.

### Which Countries have the films with the highest review score?

In [ ]:
countries_merged_df = fun.countries_reviews_merge(countries_movies_df, reviews_df)
# Group by country
country_scores = countries_merged_df.groupby('country', as_index=False)['normalized_score'].mean()

fig = px.choropleth(
    country_scores,
    locations="country",
    locationmode="country names",
    color="normalized_score",
    color_continuous_scale="Viridis",
    labels={'normalized_score': 'Avg Score (0-10)'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

This graph shows instead how there is not a correlation between Oscar wins and Rotten Tomatoes reviews. The one above in, in fact, a very different map from the ones before it, showing some African Countries, such as Libya, Somalia and South Sudan, as the Countries with the most appreciated films. The USA average is 6.23, which doesn't ut it at the bottom of the scale, but certainly it is far from the top. We can imagine the USA can produce thousands of films every year, some of which deserve awards, but many are mostly average.

COMMENTI FINALI DA FINIRE

## Which genres are most popular and acclaimed?

In [ ]:
genres_movies_df = fun.genres_movies_merge(genres_df, movies_df)
genres_oscar_df = fun.genres_oscar_merge(genres_movies_df, oscar_df)
genres_countries_df = fun.genres_countries_merge(genres_movies_df, countries_df)
genres_reviews_df = fun.genres_reviews_merge(genres_movies_df, reviews_df)

### Which genres have the most Oscar nominations? And which have the most wins?

In [ ]:
top_genres = genres_oscar_df['genre'].value_counts()
#winners_only_df = genres_oscar_df[genres_oscar_df['winner'] == True]
top_genres_win = genres_oscar_df[genres_oscar_df['winner'] == True]['genre'].value_counts()

# Sort the values in descending order (largest to smallest)
top_genres = top_genres.sort_values(ascending=True)
top_genres_win = top_genres_win.sort_values(ascending=True)
viridis_colours = plt.cm.viridis(np.linspace(0, 1, 19))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
#Donut Chart 1
axes[0].pie(top_genres, labels=top_genres.index, colors=viridis_colours, startangle=-270, wedgeprops={'width': 0.5},
        textprops={'fontsize': 7, 'fontweight': 'bold'})
#Donut Chart2
axes[1].pie(top_genres_win, labels=top_genres_win.index, colors=viridis_colours, startangle=-270, wedgeprops={'width': 0.5},
        textprops={'fontsize': 7, 'fontweight': 'bold'})
plt.tight_layout()
plt.show()

While these two plots look very similar at first glance, they actually show some differences. The top genres, drama, romance and comedies, have a very similar percentage of nominations and wins, but after them there are some differences, for example animation films have more wins tha documentaries, despite having slightly less nominations.

### Correlation between Oscar nominations and wins by genre:

In [ ]:
genre_stats_df = fun.nom_win_number(genres_oscar_df)

# Interactive bubble plot
fig = px.scatter(genre_stats_df,
                 x='Nominations',
                 y='Wins',
                 size='Nominations', # Bubble size represents the number of nominations
                 color='genre',
                 hover_data={'Nominations': True, 'Wins': True},
                 title='Interactive Bubble Plot: Nominations vs Wins by Genre')

fig.update_layout(
    margin={"r":10,"t":40,"l":10,"b":40},
    xaxis_title='Number of Nominations',
    yaxis_title='Number of Wins'
)

Differntly from the analysis we did on the Countries, here we can see that there isn't only one entry at the top of the chart, but Romance and Comedy are toghether the two top genres when it comes to Oscar Awards.

### Which genres have the films with the highest review score?

In [ ]:
# We'll make 2 Violin Plots since there are so many genres
unique_genres = sorted(genres_reviews_df['genre'].dropna().unique())
first_half = unique_genres[:9]
second_half = unique_genres[9:]
plot_df1 = genres_reviews_df[genres_reviews_df['genre'].isin(first_half)]
plot_df2 = genres_reviews_df[genres_reviews_df['genre'].isin(second_half)]

fig, axes = plt.subplots(2, 1, figsize=(15,16))
# First plot (First 9 genres) on the first axis (axes[0])
sns.violinplot(
    x='genre',
    y='normalized_score',
    data=plot_df1,
    inner='quart',
    hue='genre',
    palette='cubehelix',
    ax=axes[0]
)
axes[0].set_title('Rotten Tomatoes Scores by Genre (1–9)', fontsize=16)
axes[0].set_xlabel('Genre', fontsize=14)
axes[0].set_ylabel('Normalized Score (out of 10)', fontsize=14)
axes[0].tick_params(axis='x', rotation=45)

# Second plot (Next 9 genres) on the second axis (axes[1])
sns.violinplot(
    x='genre',
    y='normalized_score',
    data=plot_df2,
    inner='quart',
    hue='genre',
    palette='cubehelix',
    ax=axes[1]
)
axes[1].set_title('Rotten Tomatoes Scores by Genre (10–18)', fontsize=16)
axes[1].set_xlabel('Genre', fontsize=14)
axes[1].set_ylabel('Normalized Score (out of 10)', fontsize=14)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

These violin plots give us a general idea, but let's create one more plot to see the differences in scores in detail.

In [ ]:
genre_avg_scores = genres_reviews_df.groupby('genre')['normalized_score'].mean().sort_values(ascending=False)

# Bar plot
plt.figure(figsize=(12, 8))
sns.barplot(
    x=genre_avg_scores.index,
    y=genre_avg_scores.values,
    hue = genre_avg_scores.index,
    palette='viridis'
)

plt.title('Average Rotten Tomatoes Score by Genre', fontsize=16)
plt.xlabel('Genre', fontsize=14)
plt.ylabel('Average Score (out of 10)', fontsize=14)
plt.xticks(rotation=45)
plt.ylim(0, 10)
plt.tight_layout()
plt.show()

As this barplot confirms, there isn't much of a difference between the average review scores of the different genres. We can see how historical movies are the genre with the highest score, and critics appreciate action films the least, but overall they all scored similarly on Rotten Tomatoes.

### Which genres are most popular in each Country?

Since the number of Countries is quite large, we will be analyzing only the top 20 by number of films.

In [ ]:
pivot_df = fun.pivot_count_gen(genres_countries_df, reviews_df)
# Heatmap
plt.figure(figsize=(14, 8))
sns.heatmap(pivot_df,
            cmap='viridis_r',
            annot=True,
            fmt=".2f",
            linewidths=0.5)
plt.title('Average Normalized Score by Country and Genre')
plt.xlabel('Genre')
plt.ylabel('Country')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

We can immediately see some of the most evident results, such as historical films made in the Philippines, or Westerns made in Brazil being incredibly popular and usually well made, while western movies made in the Korea usually aren't very good. This could be because less films were produced for those genres in those Countries, but on average we can be sure they are perceived in a certain way.

### How has the genres popularity changed over the years?

In [ ]:
# Extract year from date
temp_gen_rev_df['year'] = temp_gen_rev_df['date'].dt.year

genres_grouped_df = (
    temp_gen_rev_df
    .groupby(['year', 'genre'])['normalized_score']
    .mean()
    .reset_index() )

fig = px.line(
    genres_grouped_df,
    x='year',
    y='normalized_score',
    color='genre',
    title="Average Normalized Score by Genre Over Time",
    labels={
        'normalized_score': 'Average Score',
        'year': 'Year',
        'genre': 'Genre' },
    markers=True
)

fig.update_traces(line=dict(width=1))  # thinner lines
fig.update_layout(
    height=500,
    title_font=dict(size=20),
    legend_title_text='Genre',
    yaxis=dict(range=[1, 10])
)
fig.show()

Calculating the mean of the scores of each genre from each year, and looking at thier trends over time, some of these results are interesting, for example we can see the slow decline of the horror genre, ........ while others, like History or War, seem to be more stable. We did in fact see earlier that these two genres are the ones with the highest score review overall, which makes sense considering how consistent they've been throughout the years.

COMMENTI FINALI

## What is the recepie to make a great film?

### What's the correlation between Oscar Wins and reviews scores?

In [ ]:
####FAI PRIMA FUNCTIONS.PY
oscar_reviews_df = fun.oscar_reviews_merged(movies_df, oscar_df, reviews_df)
#nominations_per_country = nominees_wins_df.groupby('country').size().reset_index(name='nominations')
#wins_per_country = nominees_wins_df.groupby('country')['winner'].sum().reset_index(name='wins')

fig = px.violin(
    oscar_reviews_df,###########
    y='normalized_score',          # Ratings (e.g. 'normalized_score')
    x='winner',                # Number of Oscar wins
    box=True,                      # Add box plot inside violin
    points='all',                  # Show all points within the plot
    labels={'normalized_score': 'Movie Ratings', 'oscar_wins': 'Oscar Wins'},  # Axis labels
    title="Distribution of Movie Ratings vs. Oscar Wins",
    color='winner',            # Optional: Add color by number of wins to show different win categories
)

fig.update_layout(
    margin={"r":10,"t":40,"l":10,"b":40},
    xaxis_title="Oscar Wins",
    yaxis_title="Movie Ratings (Normalized)",
)
fig.show()

### What's the correlation between duration and success?

### What's the correlation between duration, , public interest and ratings of a movie?

### Correlation between duration and Oscar Wins (FAI UN SOLO GRAFICO CON NOMINATIONS E WINS)

### Correlation between duration and Oscar nominees

In [ ]:
Lingue più usate e quindi giuste per fare un film di successo